## Example 1: sktime functionality test 1

In [ ]:
from sktime.datasets import load_airline
from sktime.forecasting.naive import NaiveForecaster
from sktime.performance_metrics.forecasting import MeanAbsolutePercentageError

print("=== Basic Forecasting ===")

y = load_airline()
print(f"Dataset shape: {y.shape}")

y_train, y_test = y[:-12], y[-12:]

forecaster = NaiveForecaster(strategy="last")
forecaster.fit(y_train)
y_pred = forecaster.predict(fh=range(1, 13))

mape = MeanAbsolutePercentageError()
score = mape(y_test, y_pred)
print(f"MAPE: {score:.2%}")

print(f"First 3 predictions: {y_pred[:3].values}")
print("Example 1 completed!")

## Example 2: sktime functionality test 2

In [ ]:
from sktime.datasets import load_airline
from sktime.forecasting.trend import TrendForecaster
from sktime.forecasting.exp_smoothing import ExponentialSmoothing
from sktime.performance_metrics.forecasting import MeanAbsolutePercentageError

print("=== Trend and Seasonal Forecasting ===")

y = load_airline()
y_train, y_test = y[:-24], y[-24:]

# Exponential smoothing
exp_smooth = ExponentialSmoothing(trend="add", seasonal="additive", sp=12)
exp_smooth.fit(y_train)
exp_pred = exp_smooth.predict(fh=range(1, 25))

mape = MeanAbsolutePercentageError()
score = mape(y_test, exp_pred)
print(f"Exponential Smoothing MAPE: {score:.2%}")

# Trend forecaster
trend = TrendForecaster()
trend.fit(y_train)
trend_pred = trend.predict(fh=range(1, 25))
trend_score = mape(y_test, trend_pred)
print(f"Trend MAPE: {trend_score:.2%}")

print("Example 2 completed!")

## Example 3: sktime functionality test 3

In [ ]:
import numpy as np
import pandas as pd
from sktime.datasets import load_airline
from sktime.transformations.series.detrend import Detrender
from sktime.transformations.series.difference import Differencer
from sktime.forecasting.trend import TrendForecaster

print("=== Time Series Transformations ===")

y = load_airline()
print(f"Original mean: {y.mean():.2f}, std: {y.std():.2f}")

# Detrend
detrend = Detrender(forecaster=TrendForecaster())
y_detrended = detrend.fit_transform(y)
print(f"Detrended mean: {y_detrended.mean():.2f}, std: {y_detrended.std():.2f}")

# Difference
diff = Differencer(lags=1)
y_diff = diff.fit_transform(y)
print(f"Differenced shape: {y_diff.shape}, mean: {y_diff.mean():.2f}")

# Seasonal difference
seasonal_diff = Differencer(lags=12)
y_seasonal_diff = seasonal_diff.fit_transform(y)
print(f"Seasonal diff mean: {y_seasonal_diff.mean():.2f}")

print("Example 3 completed!")

## Example 4: sktime functionality test 4

In [ ]:
from sktime.datasets import load_airline
from sktime.forecasting.naive import NaiveForecaster
from sktime.forecasting.exp_smoothing import ExponentialSmoothing
from sktime.forecasting.model_selection import ExpandingWindowSplitter
from sktime.performance_metrics.forecasting import mean_absolute_error
import numpy as np

print("=== Cross-Validation ===")

y = load_airline()

cv = ExpandingWindowSplitter(initial_window=60, step_length=12, fh=[1, 6, 12])

forecasters = {
    'Naive': NaiveForecaster(strategy="last"),
    'ExpSmooth': ExponentialSmoothing(sp=12)
}

for name, forecaster in forecasters.items():
    scores = []
    for train_idx, test_idx in cv.split(y):
        forecaster.fit(y.iloc[train_idx])
        pred = forecaster.predict(fh=[1, 6, 12])
        mae = mean_absolute_error(y.iloc[test_idx], pred)
        scores.append(mae)
    print(f"{name} CV MAE: {np.mean(scores):.2f} ± {np.std(scores):.2f}")

print("Example 4 completed!")

## Example 5: sktime functionality test 5

In [ ]:
from sktime.datasets import load_airline
from sktime.forecasting.naive import NaiveForecaster
from sktime.forecasting.trend import TrendForecaster
from sktime.forecasting.exp_smoothing import ExponentialSmoothing
from sktime.performance_metrics.forecasting import mean_absolute_error
import numpy as np

print("=== Multiple Model Comparison ===")

y = load_airline()
y_train, y_test = y[:-12], y[-12:]

models = {
    'Naive Last': NaiveForecaster(strategy="last"),
    'Naive Mean': NaiveForecaster(strategy="mean"),
    'Naive Drift': NaiveForecaster(strategy="drift"),
    'Trend': TrendForecaster(),
    'Exp Smoothing': ExponentialSmoothing(sp=12)
}

for name, model in models.items():
    model.fit(y_train)
    pred = model.predict(fh=range(1, 13))
    mae = mean_absolute_error(y_test, pred)
    print(f"{name}: MAE={mae:.2f}")

print("Example 5 completed!")